In [ ]:
%load_ext autoreload
%autoreload 2

In [2]:
import nnsight
@nnsight.trace
def streamer(tokens, model, max_length=80, state=None):
    # Initialize state if not provided
    if state is None:
        state = {'current_line': '', 'current_line_length': 0}

    token = tokens[-1] # only use last token

    # Decode the token
    decoded_token = model.tokenizer.decode(token).encode("unicode_escape").decode()

    if (decoded_token == '\\n') or (decoded_token == '\n'):  # Handle explicit newline tokens
        # Print the current line and reset state
        print('',flush=True)
        state['current_line'] = ''
        state['current_line_length'] = 0
    elif (decoded_token == '\n\n') or (decoded_token == '\\n\\n'):
        # Print the current line and reset state
        print('',flush=True)
        print('',flush=True)
        state['current_line'] = ''
        state['current_line_length'] = 0
    else:
        # Check if adding the token would exceed the max length
        if state['current_line_length'] + len(decoded_token) > max_length:
            print('',flush=True)
            state['current_line'] = decoded_token  # Start a new line with the current token
            state['current_line_length'] = len(decoded_token)
            print(decoded_token, flush=True, end="")  # Print ONLY the new token
        else:
            # Add a space if the line isn't empty and append the token
            if state['current_line']:
                state['current_line'] += decoded_token
            else:
                state['current_line'] = decoded_token
            state['current_line_length'] += len(decoded_token)
            print(decoded_token, flush=True, end="")  # Print ONLY the new token

    return state

/disk/u/gio/.conda/envs/retrieval/lib/python3.11/site-packages/nnsight/__init__.py:80: UserWarning: nnsight.trace is deprecated as of v0.5.0 and will be removed in a future version.
  warnings.warn(deprecation_message)


In [5]:
from nnsight import LanguageModel

qwen_model = LanguageModel("deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")

tokenizer = qwen_model.tokenizer

In [ ]:
#prompt = "What word comes next? fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, <think>\n\nOkay, so I"
prompt = "What word comes next? fish, card, autumn, web, "

In [46]:
import torch

nnsight.CONFIG.APP.REMOTE_LOGGING = False

# Initialize the state for decoding
state = {'current_line': '', 'current_line_length': 0}

all_tokens = []

with qwen_model.generate(prompt, max_new_tokens=6000, do_sample=False) as tracer:

    with tracer.all():

        # Access qwen_model output
        out = qwen_model.lm_head.output.save()

        # Apply softmax to obtain probs and save the result
        probs = torch.nn.functional.softmax(out, dim=-1)
        max_probs = torch.max(probs, dim=-1)
        tokens = max_probs.indices.cpu().tolist()
        all_tokens.append(tokens[0]).save()

        state = streamer(tokens[0], qwen_model, max_length=160, state=state)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


 need to figure out what word comes next in this sequence: fish, card, autumn, web, fish, card, autumn, web, and then it repeats. Let me look at the pattern
 here.\n\nFirst, I notice that the sequence goes fish, card, autumn, web, and then repeats. So, the first four words are fish, card, autumn, web. Then it starts
 again with fish, card, autumn, web. So, it seems like the pattern is four words repeating over and over.\n\nLet me break it down:\n\n1. fish
2. card
3. autumn
4. web
5. fish
6. card
7. autumn
8. web
9. fish
10. card
11. autumn
12. web
13. fish
14. card
15. autumn
16. web
17. fish
18. card
19. autumn
20. web
21. fish
22. card
23. autumn
24. web
25. fish
26. card
27. autumn
28. web
29. fish
30. card
31. autumn
32. web
33. fish
34. card
35. autumn
36. web
37. fish
38. card
39. autumn
40. web
41. fish
42. card
43. autumn
44. web
45. fish
46. card
47. autumn
48. web
49. fish
50. card
51. autumn
52. web
53. fish
54. card
55. autumn
56. web
57. fish
58. card
59. autumn
60. web
61

/disk/u/gio/.conda/envs/retrieval/lib/python3.11/site-packages/nnsight/intervention/interleaver.py:284: UserWarning: Execution complete but `model.lm_head.output.i3439` was not provided. This was in an Iterator at iteration 3439 so likely this iteration did not happen. If you were using `.iter[:]`, this is likely not an error.
  warnings.warn(msg)


In [47]:
len(tokenizer.decode([token[0] for token in all_tokens]))

8156

In [48]:
from nnsight import LanguageModel

nemotron_model = LanguageModel("nvidia/Nemotron-Research-Reasoning-Qwen-1.5B")

nemotron_tokenizer = nemotron_model.tokenizer

In [49]:
import torch

nnsight.CONFIG.APP.REMOTE_LOGGING = False

# Initialize the state for decoding
state = {'current_line': '', 'current_line_length': 0}

all_tokens = []

with nemotron_model.generate(prompt, max_new_tokens=6000, do_sample=False) as tracer:

    with tracer.all():

        # Access nemotron_model output
        out = nemotron_model.lm_head.output.save()

        # Apply softmax to obtain probs and save the result
        probs = torch.nn.functional.softmax(out, dim=-1)
        max_probs = torch.max(probs, dim=-1)
        tokens = max_probs.indices.cpu().tolist()
        all_tokens.append(tokens[0]).save()

        state = streamer(tokens[0], nemotron_model, max_length=160, state=state)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

 need to figure out what word comes next in the sequence provided. Let me look at the list again:\n\nfish, card, autumn, web, fish, card, autumn, web, fish,
 card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card,
 autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn,
 web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish
, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card,
 autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn,
 web, fish, card, autumn, web, fish, card, autumn

KeyboardInterrupt: 

In [30]:
tokenizer.decode([token[0] for token in all_tokens])
#all_tokens

'Okay, so I need to figure out what word comes next in the sequence provided. Let me look at the list again:\n\nfish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, autumn, web, fish, card, a